In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc matplotlib numpy pandas
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# Quantenkernel-Training

*Verbrauchsschätzung: unta ana Minut'n aufm Eagle r3 Prozessor (ACHTUNG: Des is bloß a Schätzung. Ihre Laufzeit kennt anders ausschau'n.)*
## Hintagrund
Des Tutorial zeigt, wia ma a `Qiskit-Musta` baut zum Ausrechnen von Einträg'n in ana Quantenkernel-Matrix, de fia binäre Klassifikation g'braucht werd. Fia mehra Informationa üba `Qiskit-Musta` und wia `Qiskit Serverless` verwendet werd'n kann, um's in de Cloud z'deployen fia verwaltete Ausführung, schaug'ns auf unsare [Dokumentations-Seit'n üba IBM Quantum&reg; Platform](/guides/serverless).
## Voraussetzung'n
Bevor S' mit dem Tutorial anfang'n, stöll'ns sicha, dass S' des do installiert hab'n:
- Qiskit SDK v1.0 oda späta, mit [Visualisierung](https://docs.quantum.ibm.com/api/qiskit/visualization)-Untastützung
- Qiskit Runtime v0.22 oda späta (`pip install qiskit-ibm-runtime`)
## Aufbau

In [ ]:
# General Imports and helper functions
import urllib.request

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


from qiskit.circuit import Parameter, ParameterVector, QuantumCircuit
from qiskit.circuit.library import unitary_overlap
from qiskit.primitives import StatevectorSampler
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager

from qiskit_ibm_runtime import QiskitRuntimeService, Sampler

# Download the dataset (portable across platforms)
urllib.request.urlretrieve(
    "https://raw.githubusercontent.com/qiskit-community/prototype-quantum-kernel-training/main/data/dataset_graph7.csv",
    "dataset_graph7.csv",
)


def visualize_counts(res_counts, num_qubits, num_shots):
    """Visualize the outputs from the Qiskit Sampler primitive."""
    zero_prob = res_counts.get(0, 0.0)
    top_10 = dict(
        sorted(res_counts.items(), key=lambda item: item[1], reverse=True)[
            :10
        ]
    )
    top_10.update({0: zero_prob})
    by_key = dict(sorted(top_10.items(), key=lambda item: item[0]))
    x_vals, y_vals = list(zip(*by_key.items()))
    x_vals = [bin(x_val)[2:].zfill(num_qubits) for x_val in x_vals]
    y_vals_prob = []
    for t in range(len(y_vals)):
        y_vals_prob.append(y_vals[t] / num_shots)
    y_vals = y_vals_prob
    plt.bar(x_vals, y_vals)
    plt.xticks(rotation=75)
    plt.title("Results of sampling")
    plt.xlabel("Measured bitstring")
    plt.ylabel("Probability")
    plt.show()


def get_training_data():
    """Read the training data."""
    df = pd.read_csv("dataset_graph7.csv", sep=",", header=None)
    training_data = df.values[:20, :]
    ind = np.argsort(training_data[:, -1])
    X_train = training_data[ind][:, :-1]

    return X_train

## Small-scale simulator example

In this section, we walk through the four steps of the Qiskit pattern on a seven-qubit instance of the labeling-cosets-with-error problem and evaluate a single kernel matrix entry using the `StatevectorSampler` primitive from Qiskit. A statevector simulator is exact (up to shot noise) and shows us the method end-to-end without consuming QPU time. We then repeat the same instance on real hardware in the hardware example section.

### Step 1: Map classical inputs to a quantum problem

*   Input: Training dataset.
*   Output: Abstract circuit for calculating a kernel matrix entry.

The binary classification problem we aim to solve here is referred to as "[_labeling cosets with error_](https://arxiv.org/abs/2105.03406)." The input training dataset contains a group structure, consisting of two cosets formed by a group and subgroup.
The group is taken to be $G = SU(2)^{\otimes n}$ for qubits, which is the special unitary group of
$2 \times 2$ matrices and has wide applicability in nature; e.g., the Standard Model of particle physics.
We take the (graph-stabilizer) subgroup $S_\text{graph} < G$ with $S_\text{graph} = \langle \{ X_i \otimes _{k:(k,i) \in \mathcal{E}} Z_k\} _{i \in \mathcal{V}} \} \rangle$ for a graph with edges $\mathcal{E}$ and vertices $\mathcal{V}$.
Note that the stabilizers fix a stabilizer state such that $D_s | \psi \rangle = | \psi \rangle,~ \forall s \in S_\text{graph}$.
Finally, we define two left-cosets $C_\pm = c_\pm S_\text{graph}$ by drawing two $c_\pm \in G$ at random.

For more details about the dataset and how it is generated, see [this notebook](https://github.com/qiskit-community/prototype-quantum-kernel-training/blob/main/docs/background/qkernels_and_data_w_group_structure.ipynb) from the [Quantum Kernel Training Toolkit](https://github.com/qiskit-community/prototype-quantum-kernel-training/tree/main).

We create the quantum circuit used to evaluate one entry in the kernel matrix.
The input data is used to determine the rotation angles for the circuit's parametrized gates.
For simplicity, we will use data samples `x1=14` and `x2=19`.

***Note: The dataset used in this tutorial can be downloaded [here](https://github.com/qiskit-community/prototype-quantum-kernel-training/blob/main/data/dataset_graph7.csv).***

In [2]:
# Prepare training data
X_train = get_training_data()

# Empty kernel matrix
num_samples = np.shape(X_train)[0]
kernel_matrix = np.full((num_samples, num_samples), np.nan)

# Prepare feature map for computing overlap
num_features = np.shape(X_train)[1]
num_qubits = int(num_features / 2)
entangler_map = [[0, 2], [3, 4], [2, 5], [1, 4], [2, 3], [4, 6]]
fm = QuantumCircuit(num_qubits)
training_param = Parameter("θ")
feature_params = ParameterVector("x", num_qubits * 2)
fm.ry(training_param, fm.qubits)
for cz in entangler_map:
    fm.cz(cz[0], cz[1])
for i in range(num_qubits):
    fm.rz(-2 * feature_params[2 * i + 1], i)
    fm.rx(-2 * feature_params[2 * i], i)

# Assign tunable parameter to known optimal value and set the data params for first two samples
x1 = 14
x2 = 19
unitary1 = fm.assign_parameters(list(X_train[x1]) + [np.pi / 2])
unitary2 = fm.assign_parameters(list(X_train[x2]) + [np.pi / 2])

# Create the overlap circuit
overlap_circ = unitary_overlap(unitary1, unitary2)
overlap_circ.measure_all()
overlap_circ.draw("mpl", scale=0.6, style="iqp")

<Image src="../docs/images/tutorials/quantum-kernel-training/extracted-outputs/70d6faff-9a56-44bb-b26f-f573a8c90889-0.avif" alt="Output of the previous code cell" />

## Schritt 1: Klassische Eingab'n auf a Quantenproblem abbüld'n
*   Eingab: Trainingsdatnsatz.
*   Ausgab: Abstrakta Schaltkreis zum Berechnen von am Kernelmatrix-Eintrag.

Bau'ns den Quantenschaltkreis, dea g'braucht werd zum Ausrechnen von am einzeln'n Eintrag in da Kernelmatrix. Mia brauch'n de Eingab-Dat'n zum festleg'n, wos de Rotationswinkel fia de parametrisiert'n Gates san. Mia werd'n de Dat'n-Prob'n `x1=14` und `x2=19` verwend'n.

***Achtung: Da Datnsatz, dea in dem Tutorial g'braucht werd, kann [do](https://github.com/qiskit-community/prototype-quantum-kernel-training/blob/main/data/dataset_graph7.csv) runtag'lod'n werd'n.***

In [3]:
sampler = StatevectorSampler()

# Execute and get counts
num_shots = 10_000
results = sampler.run([overlap_circ], shots=num_shots).result()
counts = results[0].data.meas.get_int_counts()

# Plot counts
visualize_counts(counts, num_qubits, num_shots)

<Image src="../docs/images/tutorials/quantum-kernel-training/extracted-outputs/step3-small-code-0.avif" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/tutorials/quantum-kernel-training/extracted-outputs/70d6faff-9a56-44bb-b26f-f573a8c90889-0.avif)

## Schritt 2: Des Problem fia Quantenhardware-Ausführung optimier'n
*   Eingab: Abstrakta Schaltkreis, net optimiert fia a bestimmts Backend
*   Ausgab: Ziel-Schaltkreis und Observable, optimiert fia de ausgsuachte QPU

Verwend'ns de `generate_preset_pass_manager`-Funktion von Qiskit zum an Optimierungslauf fia unsern Schaltkreis z'spezifizier'n, bezog'n auf de QPU, auf dea mia des Experiment lauf'n loss'n woll'n. Mia setz'n `optimization_level=3`, wos bedeut, dass mia den vordefiniert'n Pass Manager verwend'n, dea des höchste Optimierungsniveau bringt.

In [4]:
kernel_matrix[x1, x2] = counts.get(0, 0.0) / num_shots
print(f"Fidelity (simulator): {kernel_matrix[x1, x2]}")

Fidelity (simulator): 0.8261


![Output of the previous code cell](../docs/images/tutorials/quantum-kernel-training/extracted-outputs/49607b34-9723-493d-85da-bd97c1351104-0.avif)

## Schritt 3: Ausführung mit Qiskit-Primitiv'n
*   Eingab: Ziel-Schaltkreis
*   Ausgab: Quasi-Wahrscheinlichkeitsverteilung

Verwend'ns den `Sampler`-Primitiv von Qiskit Runtime zum a Quasi-Wahrscheinlichkeitsverteilung von Zuständ'n z'rekonstruier'n, de ausm Sampeln vom Schaltkreis rauskumm'n. Fürs Erstell'n von ana Kernelmatrix san mia bsondas interessiert an da Wahrscheinlichkeit, den |0>-Zustand z'mess'n.

Fia des Demo loss'n mia des auf ana QPU mit `qiskit-ibm-runtime`-Primitiv'n lauf'n. Um des auf `qiskit`-Statevector-basierte Primitiv'n lauf'n z'loss'n, ersetz'ns den Codeblock fia Qiskit IBM&reg; Runtime-Primitiv'n durch den kommentiert'n Block.

In [5]:
# ------------------------------ Step 1 ------------------------------
# Prepare training data
X_train = get_training_data()

# Empty kernel matrix
num_samples = np.shape(X_train)[0]
kernel_matrix = np.full((num_samples, num_samples), np.nan)

# Prepare feature map for computing overlap
num_features = np.shape(X_train)[1]
num_qubits = int(num_features / 2)
entangler_map = [[0, 2], [3, 4], [2, 5], [1, 4], [2, 3], [4, 6]]
fm = QuantumCircuit(num_qubits)
training_param = Parameter("θ")
feature_params = ParameterVector("x", num_qubits * 2)
fm.ry(training_param, fm.qubits)
for cz in entangler_map:
    fm.cz(cz[0], cz[1])
for i in range(num_qubits):
    fm.rz(-2 * feature_params[2 * i + 1], i)
    fm.rx(-2 * feature_params[2 * i], i)

# Assign tunable parameter to known optimal value and set the data params for first two samples
x1 = 14
x2 = 19
unitary1 = fm.assign_parameters(list(X_train[x1]) + [np.pi / 2])
unitary2 = fm.assign_parameters(list(X_train[x2]) + [np.pi / 2])

# Create the overlap circuit
overlap_circ = unitary_overlap(unitary1, unitary2)
overlap_circ.measure_all()

# ------------------------------ Step 2 ------------------------------
service = QiskitRuntimeService()
# backend = service.least_busy(
#    operational=True, simulator=False, min_num_qubits=overlap_circ.num_qubits
# )
backend = service.backend("ibm_pittsburgh")
print(f"Using backend: {backend.name}")
pm = generate_preset_pass_manager(optimization_level=3, backend=backend)
overlap_ibm = pm.run(overlap_circ)

# ------------------------------ Step 3 ------------------------------
sampler = Sampler(mode=backend)
sampler.options.environment.job_tags = ["TUT_QKT"]

num_shots = 10_000
results = sampler.run([overlap_ibm], shots=num_shots).result()
counts = results[0].data.meas.get_int_counts()
visualize_counts(counts, num_qubits, num_shots)

# ------------------------------ Step 4 ------------------------------
kernel_matrix[x1, x2] = counts.get(0, 0.0) / num_shots
print(f"Fidelity (hardware): {kernel_matrix[x1, x2]}")

Using backend: ibm_pittsburgh


<Image src="../docs/images/tutorials/quantum-kernel-training/extracted-outputs/d2f4f6cf-067e-4d53-aa04-7ca9c803d3e1-1.avif" alt="Output of the previous code cell" />

Fidelity (hardware): 0.7517


![Output of the previous code cell](../docs/images/tutorials/quantum-kernel-training/extracted-outputs/d2f4f6cf-067e-4d53-aa04-7ca9c803d3e1-0.avif)

## Schritt 4: Nachbearbeitung und des Resultat im g'wünschtn klassisch'n Format zruckgeb'n

*   Eingab: Wahrscheinlichkeitsverteilung
*   Ausgab: A einzelns Kernelmatrix-Element

Rechna'ns de Wahrscheinlichkeit zum |0> aufm Overlap-Schaltkreis z'mess'n und füll'ns de Kernelmatrix an da Position, de den Prob'n entspricht, de durch den bestimmt'n Overlap-Schaltkreis darg'stellt werd'n (Reih 15, Spalt'n 20). In dera Visualisierung zeigt dünklares Rot Fidelität'n näha an 1.0. Um de ganze Kernelmatrix auszufüll'n, müass'n mia a Quantenexperiment fia jed'n Eintrag lauf'n loss'n.